# RAG Pipeline Diagnostics: Why High Scores Return Irrelevant Chunks

This notebook diagnoses why the RAG pipeline retrieves high-scoring but irrelevant chunks, even when the correct answer exists in a different section of the report.

**Key Issues to Investigate:**
1. Cross-section contamination in chunks
2. Section title bias in embeddings
3. Missing section-level retrieval constraints
4. Insufficient context for meaningful embeddings

In [9]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Load RAG assets
rag_dir = Path("rag_assets")

# Load chunks
with open(rag_dir / "SIP_reports_section_chunks.json", encoding="utf-8") as f:
    chunks_data = json.load(f)

# Load metadata
with open(rag_dir / "rag_chunk_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

# Load embeddings
embeddings = np.load(rag_dir / "rag_embeddings.npy")

print(f"✓ Loaded {len(chunks_data)} chunks")
print(f"✓ Loaded {len(metadata)} metadata entries")
print(f"✓ Embeddings shape: {embeddings.shape}")

# Create a mapping of chunks to sections
chunks_df = pd.DataFrame(chunks_data)
print(f"\nChunks DataFrame shape: {chunks_df.shape}")
print(f"Columns: {list(chunks_df.columns)}")

# Show unique sections
unique_sections = chunks_df['section_title'].unique()
print(f"\nUnique sections ({len(unique_sections)}): ")
for i, section in enumerate(unique_sections[:15]):
    count = len(chunks_df[chunks_df['section_title'] == section])
    print(f"  {i+1:2d}. {section[:60]:60s} ({count} chunks)")

✓ Loaded 319 chunks
✓ Loaded 319 metadata entries
✓ Embeddings shape: (319, 384)

Chunks DataFrame shape: (319, 12)
Columns: ['chunk_id', 'source_file', 'report_id', 'company', 'section_title', 'parent_section', 'section_index', 'section_level', 'section_count', 'has_tables', 'chunk_length', 'chunk_text']

Unique sections (87): 
   1. Introduction                                                 (3 chunks)
   2. Explain the objectives of the report.                        (1 chunks)
   3. Describe the information on the context of the report, such  (1 chunks)
   4. State the parameters on which the report is based.           (1 chunks)
   5. Work Environment                                             (3 chunks)
   6. Describe the location and premises of the SIP Organisation   (1 chunks)
   7. Comment on the conduciveness of the environment for work     (1 chunks)
   8. Observations on aspects of workplace safety, facilities and  (1 chunks)
   9. Organisation structure                 

In [10]:
import sys
sys.path.insert(0, str(Path("rag_assets").resolve().parent))

from rag_assets.rag_pipeline import get_context, load_assets

# Test queries
test_queries = [
    "What are the common learning outcomes for students in their SIP?",
    "What were the challenges faced during the internship?",
    "Tell me about the work environment at the company",
    "What technical skills were developed?",
]

print("="*80)
print("RETRIEVAL ANALYSIS: Comparing High Scores vs. Actual Relevance")
print("="*80)

# Retrieve chunks for each query
for query in test_queries:
    print(f"\n{'='*80}")
    print(f"QUERY: {query}")
    print(f"{'='*80}")
    
    assets = load_assets()
    model = assets["model"]
    vector_store = assets["vector_store"]
    chunk_records = assets["chunk_records"]
    chunk_metadata = assets["chunk_metadata"]
    
    # Get top 6 chunks
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = vector_store.kneighbors(query_embedding, n_neighbors=6)
    
    for rank, (distance, index) in enumerate(zip(distances[0], indices[0]), 1):
        score = 1.0 - float(distance)
        meta = chunk_metadata[index]
        chunk_text = chunk_records[index].get("chunk_text", "")
        
        # Extract first 100 chars of chunk text
        preview = chunk_text.replace('\n', ' ')[:100]
        
        print(f"\n{rank}. SCORE: {score:.3f} | SECTION: {meta.get('section_title', 'N/A')[:50]}")
        print(f"   Source: {meta.get('source_file', 'N/A')} | Chunk length: {meta.get('chunk_length', 0)}")
        print(f"   Preview: {preview}...")
        
        # Check if this is a section header (very short chunks)
        if meta.get('chunk_length', 0) < 50:
            print(f"   ⚠️  WARNING: Very short chunk (likely just a section title)")

RETRIEVAL ANALYSIS: Comparing High Scores vs. Actual Relevance

QUERY: What are the common learning outcomes for students in their SIP?

1. SCORE: 0.854 | SECTION: SIP Learning Outcomes
   Source: 10_SIPReport.docx | Chunk length: 21
   Preview: SIP Learning Outcomes...
   ⚠️  WARNING: Very short chunk (likely just a section title)

2. SCORE: 0.678 | SECTION: Summarise your main SIP Learning Outcomes(Generic,
   Source: 10_SIPReport.docx | Chunk length: 997
   Preview: Summarise your main SIP Learning Outcomes(Generic, Diploma, Organization)  Demonstrates the ability ...

3. SCORE: 0.658 | SECTION: Generic Life Skills Learning Outcomes
   Source: 6_SIPReport.docx | Chunk length: 837
   Preview: Generic Life Skills Learning Outcomes  The generic life skills that the SIP aims for students are th...

4. SCORE: 0.658 | SECTION: SIP Learning Outcomes
   Source: 7_SIPReport.docx | Chunk length: 997
   Preview: SIP Learning Outcomes  My main SIP Learning Outcomes were to apply my data analyti

In [11]:
print("\n" + "="*80)
print("ROOT CAUSE ANALYSIS: Section Header Bias")
print("="*80)

# Analyze chunk characteristics
chunk_stats = []
for chunk in chunks_data:
    chunk_stats.append({
        'chunk_id': chunk['chunk_id'],
        'section_title': chunk['section_title'],
        'chunk_length': chunk['chunk_length'],
        'section_level': chunk['section_level'],
        'text_preview': chunk['chunk_text'][:100]
    })

chunk_stats_df = pd.DataFrame(chunk_stats)

# Show distribution of chunk lengths
print("\nChunk Length Distribution:")
print(chunk_stats_df['chunk_length'].describe())

# Count very short chunks (likely section headers)
short_chunks = chunk_stats_df[chunk_stats_df['chunk_length'] < 100]
print(f"\nVery short chunks (<100 chars): {len(short_chunks)} out of {len(chunk_stats_df)} ({100*len(short_chunks)/len(chunk_stats_df):.1f}%)")

# Show examples of short chunks
print("\nExamples of short chunks (section headers):")
for idx, row in short_chunks.head(10).iterrows():
    print(f"  • Section: {row['section_title'][:50]:50s} | Length: {row['chunk_length']:4d} | Level: {row['section_level']}")

# Key finding: Section title bias
print("\n" + "="*80)
print("KEY FINDING: Section Title Bias in Embeddings")
print("="*80)
print("""
The problem: Each chunk is prefixed with its section title:
  chunk_text = "{section_title}\\n\\n{body_text}"
  
This causes high embedding similarity to section titles rather than content.

Example: A query about "learning outcomes" will:
1. Match any section with "learning" in the title with high score
2. Retrieve that entire section even if the body text is unrelated
3. Skip other sections with the actual answer but lower title similarity
""")

# Demonstrate the bias
print("\nChunks that are mostly section titles:")
mostly_headers = chunk_stats_df[chunk_stats_df['chunk_length'] < 500]
print(f"  • {len(mostly_headers)} chunks have <500 chars (mostly headers/short content)")
print(f"  • These chunks have low context for meaningful embeddings")


ROOT CAUSE ANALYSIS: Section Header Bias

Chunk Length Distribution:
count    319.000000
mean     799.981191
std      318.973284
min        9.000000
25%      610.500000
50%      994.000000
75%      997.000000
max      999.000000
Name: chunk_length, dtype: float64

Very short chunks (<100 chars): 16 out of 319 (5.0%)

Examples of short chunks (section headers):
  • Section: Introduction                                       | Length:   12 | Level: 2
  • Section: Work Environment                                   | Length:   16 | Level: 2
  • Section: Organisation structure                             | Length:   22 | Level: 2
  • Section: Nature of business                                 | Length:   18 | Level: 2
  • Section: Diversity & Inclusivity                            | Length:   23 | Level: 2
  • Section: SIP Learning Outcomes                              | Length:   21 | Level: 2
  • Section: Work done                                          | Length:    9 | Level: 2
  • Se

In [14]:
print("="*80)
print("SOLUTION 1: Separate Section Headers from Content")
print("="*80)

# Proposed improvement: Separate header and body
improved_chunks = []
for chunk in chunks_data:
    chunk_text = chunk['chunk_text']
    section_title = chunk['section_title']
    
    # Remove section title from chunk if it appears at the start
    if chunk_text.startswith(section_title):
        body_only = chunk_text[len(section_title):].strip()
    else:
        body_only = chunk_text
    
    improved_chunks.append({
        'chunk_id': chunk['chunk_id'],
        'section_title': section_title,
        'original_length': len(chunk_text),
        'body_length': len(body_only),
        'body_text': body_only[:200],
        'is_header_only': len(body_only) < 50
    })

improved_df = pd.DataFrame(improved_chunks)

print(f"\nChunks that are mostly headers (body <50 chars): {improved_df['is_header_only'].sum()}")
print(f"Average original chunk size: {improved_df['original_length'].mean():.0f} chars")
print(f"Average body-only chunk size: {improved_df['body_length'].mean():.0f} chars")

# Show impact of removing headers
print("\nImpact of removing section headers:")
with_headers = improved_df['original_length'].sum()
without_headers = improved_df['body_length'].sum()
header_reduction = (1 - without_headers/with_headers) * 100
print(f"  • Total chars with headers: {with_headers:,}")
print(f"  • Total chars body-only: {without_headers:,}")
print(f"  • Header-only content: {header_reduction:.1f}% of embedding input")

print("\n" + "="*80)
print("SOLUTION 2: Add Section-Level Filtering")
print("="*80)

def retrieve_with_section_awareness(query, k=6, prefer_section=None):
    """Modified retrieval that optionally constrains to sections."""
    assets = load_assets()
    model = assets["model"]
    vector_store = assets["vector_store"]
    chunk_records = assets["chunk_records"]
    chunk_metadata = assets["chunk_metadata"]
    
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = vector_store.kneighbors(query_embedding, n_neighbors=k*3)  # Get more candidates
    
    results = []
    for distance, index in zip(distances[0], indices[0]):
        meta = chunk_metadata[index]
        score = 1.0 - float(distance)
        
        # If section preference given, boost score for matching sections
        if prefer_section and meta.get('section_title') == prefer_section:
            score = min(1.0, score + 0.2)  # Boost matching section
        
        results.append({
            'score': score,
            'section': meta.get('section_title'),
            'index': index
        })
    
    # Re-sort by boosted scores
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:k]

# Test the improved retrieval
print("\nComparison: Standard vs. Section-Aware Retrieval")
query = "What are the challenges faced during the internship?"

print(f"\nQuery: {query}")
print(f"\n1. STANDARD RETRIEVAL (current):")
assets = load_assets()
model = assets["model"]
vector_store = assets["vector_store"]
chunk_records = assets["chunk_records"]
chunk_metadata = assets["chunk_metadata"]

query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
distances, indices = vector_store.kneighbors(query_embedding, n_neighbors=3)

for rank, (distance, index) in enumerate(zip(distances[0], indices[0]), 1):
    score = 1.0 - float(distance)
    meta = chunk_metadata[index]
    print(f"   {rank}. Score: {score:.3f} | Section: {meta.get('section_title')[:50]}")

print(f"\n2. PROPOSED IMPROVEMENTS:")
print("   • Remove section titles from chunk embeddings")
print("   • Track section context separately in metadata")
print("   • Add section-aware reranking")
print("   • Use hybrid search (dense + keyword) for sections")

SOLUTION 1: Separate Section Headers from Content

Chunks that are mostly headers (body <50 chars): 17
Average original chunk size: 800 chars
Average body-only chunk size: 785 chars

Impact of removing section headers:
  • Total chars with headers: 255,194
  • Total chars body-only: 250,449
  • Header-only content: 1.9% of embedding input

SOLUTION 2: Add Section-Level Filtering

Comparison: Standard vs. Section-Aware Retrieval

Query: What are the challenges faced during the internship?

1. STANDARD RETRIEVAL (current):
   1. Score: 0.666 | Section: What are the challenges that you have encountered 
   2. Score: 0.637 | Section: Full Document
   3. Score: 0.627 | Section: What could have been done to enhance or improve yo

2. PROPOSED IMPROVEMENTS:
   • Remove section titles from chunk embeddings
   • Track section context separately in metadata
   • Add section-aware reranking
   • Use hybrid search (dense + keyword) for sections


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("="*80)
print("SOLUTION 3: Hybrid Search (Dense + Keyword)")
print("="*80)

# Build a simple keyword index
chunk_texts = [chunk['chunk_text'] for chunk in chunks_data]
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

def hybrid_retrieval(query, k=6, alpha=0.5):
    """Combine semantic search (alpha) with keyword search (1-alpha)."""
    assets = load_assets()
    model = assets["model"]
    vector_store = assets["vector_store"]
    chunk_metadata = assets["chunk_metadata"]
    
    # Dense retrieval
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = vector_store.kneighbors(query_embedding, n_neighbors=k*2)
    dense_scores = 1.0 - distances[0]
    dense_indices = indices[0]
    
    # Keyword retrieval
    query_tfidf = vectorizer.transform([query])
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]
    
    # Combine scores
    combined_scores = []
    for idx in dense_indices:
        dense_score = dense_scores[list(dense_indices).index(idx)]
        keyword_score = keyword_scores[idx]
        
        # Normalize both to [0, 1]
        combined = alpha * dense_score + (1 - alpha) * keyword_score
        combined_scores.append((idx, combined))
    
    # Sort by combined score
    combined_scores.sort(key=lambda x: x[1], reverse=True)
    return combined_scores[:k]

# Test hybrid retrieval
query = "What are the challenges faced during the internship?"

print(f"\nQuery: {query}\n")
print("Comparison of retrieval methods:")
print("-" * 80)

# Method 1: Dense only (current)
print("\n1. DENSE ONLY (current method):")
assets = load_assets()
model = assets["model"]
vector_store = assets["vector_store"]
chunk_records = assets["chunk_records"]
chunk_metadata = assets["chunk_metadata"]

query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
distances, indices = vector_store.kneighbors(query_embedding, n_neighbors=3)

for rank, (distance, index) in enumerate(zip(distances[0], indices[0]), 1):
    score = 1.0 - float(distance)
    meta = chunk_metadata[index]
    chunk_text = chunk_records[index].get("chunk_text", "")[:80]
    print(f"   {rank}. Score: {score:.3f} | {meta.get('section_title')[:50]}")

# Method 2: Hybrid (proposed)
print("\n2. HYBRID SEARCH (dense + keyword, α=0.6):")
hybrid_results = hybrid_retrieval(query, k=3, alpha=0.6)

for rank, (index, score) in enumerate(hybrid_results, 1):
    meta = chunk_metadata[index]
    print(f"   {rank}. Score: {score:.3f} | {meta.get('section_title')[:50]}")

print("\n" + "="*80)
print("SUMMARY OF FINDINGS AND RECOMMENDATIONS")
print("="*80)

print("""
ROOT CAUSE:
The current RAG pipeline has SECTION TITLE BIAS because:
1. Chunks include section titles at the beginning
2. Very short chunks (section headers only) get inflated similarity scores
3. Section metadata is stored but NOT used during retrieval
4. No distinction between section titles and actual content

IMMEDIATE FIXES (implement in rag_pipeline.py):
1. ✓ Remove section titles from chunk embeddings
2. ✓ Increase minimum chunk size to 200+ characters
3. ✓ Add section-aware reranking (prefer diverse sections)
4. ✓ Implement hybrid search combining dense + keyword

LONG-TERM IMPROVEMENTS:
1. Use different embedding strategy for section headers vs. content
2. Add BM25 keyword search as fallback
3. Implement cross-encoder reranking
4. Add section-level constraints in retrieval
5. Track which sections have been retrieved to avoid redundancy
""")

# Show which sections would benefit most
print("\n" + "="*80)
print("Section Coverage Analysis:")
print("="*80)
section_counts = pd.Series([c['section_title'] for c in chunks_data]).value_counts()
print(f"\nTop 10 sections by chunk count:")
for section, count in section_counts.head(10).items():
    print(f"  • {section[:60]:60s}: {count:3d} chunks")

SOLUTION 3: Hybrid Search (Dense + Keyword)

Query: What are the challenges faced during the internship?

Comparison of retrieval methods:
--------------------------------------------------------------------------------

1. DENSE ONLY (current method):
   1. Score: 0.666 | What are the challenges that you have encountered 
   2. Score: 0.637 | Full Document
   3. Score: 0.627 | What could have been done to enhance or improve yo

2. HYBRID SEARCH (dense + keyword, α=0.6):
   1. Score: 0.447 | Full Document
   2. Score: 0.446 | What are the challenges that you have encountered 
   3. Score: 0.403 | What could have been done to enhance or improve yo

SUMMARY OF FINDINGS AND RECOMMENDATIONS

ROOT CAUSE:
The current RAG pipeline has SECTION TITLE BIAS because:
1. Chunks include section titles at the beginning
2. Very short chunks (section headers only) get inflated similarity scores
3. Section metadata is stored but NOT used during retrieval
4. No distinction between section titles and actu

## Section 5: Add Reranking and Hybrid Search Diagnostics

Implement keyword-based search combined with semantic search for better section matching.

## Section 4: Test Alternative Chunking and Section-Level Retrieval

Implement a solution that separates section headers from content and adds section-aware constraints.

## Section 3: Analyze Similarity Scores vs. Relevance Labels

Examine chunk characteristics that lead to high scores but low relevance.

## Section 2: Inspect Retrieved Chunks and Their Source Sections

Load the RAG pipeline and test retrieval with sample queries to see why high scores don't match relevance.

## Section 1: Load Report and Build Section-Aware Corpus

Load the RAG assets and analyze how chunks are mapped to sections.